# 03 — Machine Learning & Dynamic Recommendation Evaluation
## AI-Powered Natural-Language Product Recommendation System

This notebook covers the machine learning modeling, label leakage prevention, and technical evaluation:
1. **Critical ML Problem — Label Leakage Prevention**: Why predicting target items from synthetic requirement text produces invalid 100% artificial shortcuts.
2. **Behavioral Classification Models**:
   * Baseline 1: Logistic Regression
   * Baseline 2: Random Forest Classifier
   * Primary Model: XGBoost Classifier
3. **Classification Evaluation**: Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC on an 80/20 Stratified Test Set.
4. **Pure Recommendation Evaluation**: Precision@K, Recall@K, Hit Rate@K, NDCG@K, and MRR@K on held-out user query interactions.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from src.data_preprocessing import load_customer_history, load_product_catalog, generate_synthetic_interactions
from src.ml_model import CustomerMLModel
from src.semantic_recommender import SemanticRecommender
from src.ranking import HybridRecommender
from src.evaluation import evaluate_recommender_on_interactions

sns.set_theme(style="whitegrid")

## 1. Machine Learning Training on Independent Behavioral Features

In [ ]:
history_df = load_customer_history('../data/customer_history.csv')
ml_system = CustomerMLModel()
ml_results = ml_system.train_and_evaluate(history_df, test_size=0.20, random_state=42)

# Display tabular model metrics
metrics_df = pd.DataFrame([
    {
        "Model": name,
        "Accuracy (%)": f"{vals['accuracy']*100:.2f}%",
        "F1 Score (%)": f"{vals['f1_score']*100:.2f}%",
        "Precision (%)": f"{vals['precision']*100:.2f}%",
        "Recall (%)": f"{vals['recall']*100:.2f}%",
        "Multiclass ROC-AUC": f"{vals.get('roc_auc', 0.5):.4f}" if vals.get('roc_auc') else "N/A"
    }
    for name, vals in ml_results["metrics"].items()
])
metrics_df

## 2. Confusion Matrix Heatmap (XGBoost)

In [ ]:
cm = ml_results["metrics"]["XGBoost"]["confusion_matrix"]
classes = ml_results["classes"]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Multi-Class Confusion Matrix — XGBoost Classifier', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Product Class')
plt.ylabel('Actual Product Class')
plt.xticks(rotation=45)
plt.show()

## 3. Feature Importance (XGBoost)

In [ ]:
feat_imp = ml_results["metrics"]["XGBoost"].get("feature_importance", {})
if feat_imp:
    feat_series = pd.Series(feat_imp).sort_values(ascending=True)
    plt.figure(figsize=(10, 6))
    feat_series.plot(kind='barh', color='#3B82F6')
    plt.title('XGBoost Feature Importance (Demographic & Behavioral Signals)', fontsize=14, fontweight='bold')
    plt.xlabel('Importance')
    plt.show()

## 4. Pure Recommendation Ranking Metrics Evaluation (Top 1, Top 3, Top 5)

In [ ]:
catalog_df = load_product_catalog('../data/product_catalog.csv')
semantic_engine = SemanticRecommender()
semantic_engine.fit_catalog(catalog_df)

hybrid_engine = HybridRecommender(
    catalog_df=catalog_df,
    semantic_engine=semantic_engine,
    ml_model=ml_system
)

# Generate or load held-out interaction dataset
interactions_path = '../data/customer_interactions.csv'
if os.path.exists(interactions_path):
    interactions_df = pd.read_csv(interactions_path)
else:
    interactions_df = generate_synthetic_interactions(catalog_df, num_interactions=500)

rec_eval = evaluate_recommender_on_interactions(hybrid_engine, interactions_df, k_values=[1, 3, 5])

print("=== Recommendation Ranking Metrics on Held-Out Test Set ===")
summary = rec_eval["summary"]
table_data = [
    {"Recommendation Depth": "Top 1", "Precision": f"{summary['Precision@1']*100:.2f}%", "Recall": f"{summary['Recall@1']*100:.2f}%", "Hit Rate": f"{summary['Hit_Rate@1']*100:.2f}%", "NDCG Score": f"{summary['NDCG@1']:.4f}"},
    {"Recommendation Depth": "Top 3", "Precision": f"{summary['Precision@3']*100:.2f}%", "Recall": f"{summary['Recall@3']*100:.2f}%", "Hit Rate": f"{summary['Hit_Rate@3']*100:.2f}%", "NDCG Score": f"{summary['NDCG@3']:.4f}"},
    {"Recommendation Depth": "Top 5", "Precision": f"{summary['Precision@5']*100:.2f}%", "Recall": f"{summary['Recall@5']*100:.2f}%", "Hit Rate": f"{summary['Hit_Rate@5']*100:.2f}%", "NDCG Score": f"{summary['NDCG@5']:.4f}"}
]
pd.DataFrame(table_data)

In [ ]:
# Plotting Recommendation Metrics Growth across Top-N Results
summary = rec_eval["summary"]
plot_rec_df = pd.DataFrame({
    "Results Depth": ["Top 1", "Top 3", "Top 5"],
    "Recall": [summary["Recall@1"], summary["Recall@3"], summary["Recall@5"]],
    "Hit Rate": [summary["Hit_Rate@1"], summary["Hit_Rate@3"], summary["Hit_Rate@5"]],
    "NDCG Score": [summary["NDCG@1"], summary["NDCG@3"], summary["NDCG@5"]]
})

plot_rec_df.set_index("Results Depth").plot(kind='bar', figsize=(10, 5), color=['#3B82F6', '#10B981', '#8B5CF6'])
plt.title('Recommendation Metrics across Top-N Results', fontsize=14, fontweight='bold')
plt.ylabel('Score (0.0 to 1.0)')
plt.ylim(0, 1.15)
plt.xticks(rotation=0)
plt.legend(title='Metric')
plt.show()